In [ ]:
# Use ticker for company name search
#============================================================
# CELL 1 — Install packages
# ============================================================

!pip -q install yfinance pandas requests tqdm

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


Consider BigQuery due to DOC API is suitable for manageable article samples and recent timeline analysis. Its article-list output is capped at 250 results per request; larger historical datasets generally require GDELT’s downloadable files or Google BigQuery tables.

In [ ]:
# ============================================================
# CELL 2 — Imports and configuration
# ============================================================

from __future__ import annotations

import re
import time
from datetime import datetime, timedelta
from pathlib import Path
from typing import Optional

import pandas as pd
import requests
import yfinance as yf
from tqdm.auto import tqdm


GDELT_DOC_ENDPOINT = "https://api.gdeltproject.org/api/v2/doc/doc"

In [ ]:
# ============================================================
# CELL 3 — Resolve stock ticker to company information
# ============================================================

def resolve_company_from_ticker(ticker: str) -> dict:
    """
    Resolve a stock ticker into company metadata using yfinance Search.

    Parameters
    ----------
    ticker : str
        Stock ticker, such as AAPL, MSFT, TSLA, or JPM.

    Returns
    -------
    dict
        Company metadata containing ticker, company name,
        exchange, asset type, and Yahoo Finance result.
    """
    ticker = ticker.strip().upper()

    if not ticker:
        raise ValueError("Ticker cannot be empty.")

    search = yf.Search(
        ticker,
        max_results=10,
        news_count=0,
        include_research=False,
        raise_errors=True
    )

    quotes = search.quotes or []

    if not quotes:
        raise ValueError(f"No Yahoo Finance results found for ticker: {ticker}")

    # Prefer an exact ticker match.
    exact_matches = [
        quote for quote in quotes
        if str(quote.get("symbol", "")).upper() == ticker
    ]

    selected = exact_matches[0] if exact_matches else quotes[0]

    company_name = (
        selected.get("longname")
        or selected.get("shortname")
        or selected.get("name")
    )

    if not company_name:
        raise ValueError(
            f"The ticker {ticker} was found, but its company name "
            "could not be determined."
        )

    return {
        "ticker": ticker,
        "company_name": company_name,
        "short_name": selected.get("shortname"),
        "exchange": selected.get("exchange"),
        "quote_type": selected.get("quoteType"),
        "yahoo_result": selected
    }

In [ ]:
# ============================================================
# CELL 4 — Query-building functions
# ============================================================

def clean_company_name(company_name: str) -> str:
    """
    Remove selected legal suffixes to create a shorter search alias.

    Example:
        'Apple Inc.' -> 'Apple'
        'Microsoft Corporation' -> 'Microsoft'
    """
    suffix_pattern = (
        r",?\s+("
        r"incorporated|inc\.?|corporation|corp\.?|company|co\.?|"
        r"limited|ltd\.?|plc|holdings?|group|sa|ag|nv"
        r")\.?$"
    )

    cleaned = re.sub(
        suffix_pattern,
        "",
        company_name.strip(),
        flags=re.IGNORECASE
    )

    return cleaned.strip(" ,.-")


def escape_gdelt_phrase(value: str) -> str:
    """Escape quotation marks before inserting text into a GDELT query."""
    return value.replace('"', '\\"').strip()


def build_company_query(
    ticker: str,
    company_name: str,
    aliases: Optional[list[str]] = None,
    additional_terms: Optional[list[str]] = None,
    include_ticker: bool = False
) -> str:
    """
    Build a GDELT full-text query for a company.

    The company name is used as the primary search term. Additional
    aliases can improve recall for companies frequently referenced
    by brand names rather than legal names.

    Examples of aliases:
        Alphabet Inc. -> Google
        Meta Platforms, Inc. -> Facebook, Instagram, WhatsApp
        JPMorgan Chase & Co. -> JPMorgan, Chase
    """
    aliases = aliases or []
    additional_terms = additional_terms or []

    candidate_names = [
        company_name,
        clean_company_name(company_name),
        *aliases
    ]

    # Preserve order while removing empty values and duplicates.
    unique_names = []
    seen = set()

    for name in candidate_names:
        if not name:
            continue

        normalized = name.strip().lower()

        if normalized not in seen:
            seen.add(normalized)
            unique_names.append(name.strip())

    company_terms = [
        f'"{escape_gdelt_phrase(name)}"'
        for name in unique_names
    ]

    # Tickers are disabled by default because many are ambiguous words.
    if include_ticker:
        company_terms.append(f'"{escape_gdelt_phrase(ticker.upper())}"')

    company_expression = (
        company_terms[0]
        if len(company_terms) == 1
        else f"({' OR '.join(company_terms)})"
    )

    if additional_terms:
        topic_terms = [
            f'"{escape_gdelt_phrase(term)}"'
            for term in additional_terms
            if term and term.strip()
        ]

        if topic_terms:
            topic_expression = (
                topic_terms[0]
                if len(topic_terms) == 1
                else f"({' OR '.join(topic_terms)})"
            )

            company_expression = (
                f"{company_expression} AND {topic_expression}"
            )

    return company_expression

In [ ]:
# ============================================================
# CELL 5 — Date-window and API helper functions
# ============================================================

def validate_date_range(
    start_date: str,
    end_date: str
) -> tuple[pd.Timestamp, pd.Timestamp]:
    """
    Validate dates supplied in YYYY-MM-DD format.
    """
    start = pd.to_datetime(start_date, format="%Y-%m-%d")
    end = pd.to_datetime(end_date, format="%Y-%m-%d")

    if start > end:
        raise ValueError("start_date must be earlier than or equal to end_date.")

    return start, end


def create_date_windows(
    start_date: str,
    end_date: str,
    window_days: int = 7
) -> list[tuple[pd.Timestamp, pd.Timestamp]]:
    """
    Divide a date range into smaller windows.

    Smaller windows reduce the likelihood that the API's result limit
    will omit a large share of articles during high-volume periods.
    """
    if window_days < 1:
        raise ValueError("window_days must be at least 1.")

    start, end = validate_date_range(start_date, end_date)
    windows = []

    current_start = start

    while current_start <= end:
        current_end = min(
            current_start + pd.Timedelta(days=window_days) - pd.Timedelta(seconds=1),
            end + pd.Timedelta(days=1) - pd.Timedelta(seconds=1)
        )

        windows.append((current_start, current_end))
        current_start = current_end + pd.Timedelta(seconds=1)

    return windows


def format_gdelt_datetime(value: pd.Timestamp) -> str:
    """
    Convert a timestamp into GDELT's YYYYMMDDHHMMSS format.
    """
    return value.strftime("%Y%m%d%H%M%S")


def request_gdelt_json(
    params: dict,
    max_retries: int = 4,
    timeout: int = 60
) -> dict:
    """
    Send a GDELT API request with retry handling.
    """
    headers = {
        "User-Agent": (
            "Academic-GDELT-Company-Research/1.0 "
            "(Python requests; noncommercial research)"
        )
    }

    last_error = None

    for attempt in range(max_retries):
        try:
            response = requests.get(
                GDELT_DOC_ENDPOINT,
                params=params,
                headers=headers,
                timeout=timeout
            )

            # Retry temporary server and rate-limit errors.
            if response.status_code in {429, 500, 502, 503, 504}:
                wait_seconds = 2 ** attempt
                time.sleep(wait_seconds)
                continue

            response.raise_for_status()

            # GDELT occasionally returns an empty response.
            if not response.text.strip():
                return {}

            return response.json()

        except (
            requests.RequestException,
            ValueError
        ) as error:
            last_error = error

            if attempt < max_retries - 1:
                time.sleep(2 ** attempt)
            else:
                raise RuntimeError(
                    f"GDELT request failed after {max_retries} attempts."
                ) from error

    raise RuntimeError("GDELT request failed.") from last_error

In [ ]:
# ============================================================
# CELL 6 — Main GDELT article extraction function
# ============================================================

def get_gdelt_company_articles(
    ticker: str,
    start_date: str,
    end_date: str,
    aliases: Optional[list[str]] = None,
    additional_terms: Optional[list[str]] = None,
    source_country: Optional[str] = None,
    source_language: Optional[str] = None,
    include_ticker_in_query: bool = False,
    window_days: int = 7,
    max_records_per_window: int = 250,
    request_delay: float = 1.0
) -> tuple[pd.DataFrame, dict]:
    """
    Retrieve GDELT news articles for a selected company and time period.

    Parameters
    ----------
    ticker : str
        Stock ticker, such as AAPL.
    start_date, end_date : str
        Date range in YYYY-MM-DD format.
    aliases : list[str], optional
        Brand names or common company-name variants.
    additional_terms : list[str], optional
        Optional topic filters such as crisis, lawsuit, fraud, recall,
        cybersecurity, ESG, or reputation.
    source_country : str, optional
        GDELT source-country code, such as US, GB, or CA.
    source_language : str, optional
        GDELT source-language value, such as English.
    include_ticker_in_query : bool
        Include the ticker as a search phrase. False is safer because
        many tickers are ambiguous.
    window_days : int
        Number of days included in each API request.
    max_records_per_window : int
        Maximum articles returned per request. GDELT DOC ArticleList
        requests should not exceed the API-supported maximum.
    request_delay : float
        Delay between requests to reduce rate-limit risk.

    Returns
    -------
    articles_df : pandas.DataFrame
        Retrieved and standardized articles.
    metadata : dict
        Search configuration and company-resolution metadata.
    """
    company = resolve_company_from_ticker(ticker)

    query = build_company_query(
        ticker=company["ticker"],
        company_name=company["company_name"],
        aliases=aliases,
        additional_terms=additional_terms,
        include_ticker=include_ticker_in_query
    )

    query_filters = [query]

    if source_country:
        query_filters.append(f"sourcecountry:{source_country.upper()}")

    if source_language:
        query_filters.append(f"sourcelang:{source_language}")

    final_query = " ".join(query_filters)

    windows = create_date_windows(
        start_date=start_date,
        end_date=end_date,
        window_days=window_days
    )

    all_articles = []
    request_log = []

    for window_start, window_end in tqdm(
        windows,
        desc=f"Retrieving GDELT articles for {company['ticker']}"
    ):
        params = {
            "query": final_query,
            "mode": "artlist",
            "format": "json",
            "maxrecords": min(max_records_per_window, 250),
            "sort": "datedesc",
            "startdatetime": format_gdelt_datetime(window_start),
            "enddatetime": format_gdelt_datetime(window_end)
        }

        try:
            payload = request_gdelt_json(params)
            articles = payload.get("articles", []) if payload else []

            for article in articles:
                record = article.copy()
                record["ticker"] = company["ticker"]
                record["company_name"] = company["company_name"]
                record["search_query"] = final_query
                record["query_window_start"] = window_start
                record["query_window_end"] = window_end
                all_articles.append(record)

            request_log.append({
                "window_start": window_start,
                "window_end": window_end,
                "records_returned": len(articles),
                "status": "success"
            })

        except Exception as error:
            request_log.append({
                "window_start": window_start,
                "window_end": window_end,
                "records_returned": 0,
                "status": "failed",
                "error": str(error)
            })

            print(
                f"Warning: request failed for "
                f"{window_start.date()} through {window_end.date()}: {error}"
            )

        time.sleep(request_delay)

    articles_df = pd.DataFrame(all_articles)

    if not articles_df.empty:
        # Standardize publication timestamp.
        if "seendate" in articles_df.columns:
            articles_df["seendate"] = pd.to_datetime(
                articles_df["seendate"],
                errors="coerce",
                utc=True
            )

        # Remove duplicate articles returned across adjacent query windows.
        if "url" in articles_df.columns:
            articles_df = articles_df.drop_duplicates(
                subset=["url"],
                keep="first"
            )
        elif {"title", "seendate"}.issubset(articles_df.columns):
            articles_df = articles_df.drop_duplicates(
                subset=["title", "seendate"],
                keep="first"
            )

        preferred_columns = [
            "ticker",
            "company_name",
            "seendate",
            "title",
            "url",
            "domain",
            "language",
            "sourcecountry",
            "socialimage",
            "search_query",
            "query_window_start",
            "query_window_end"
        ]

        existing_preferred = [
            column for column in preferred_columns
            if column in articles_df.columns
        ]

        remaining_columns = [
            column for column in articles_df.columns
            if column not in existing_preferred
        ]

        articles_df = articles_df[
            existing_preferred + remaining_columns
        ]

        if "seendate" in articles_df.columns:
            articles_df = articles_df.sort_values(
                "seendate",
                ascending=True
            )

        articles_df = articles_df.reset_index(drop=True)

    metadata = {
        "ticker": company["ticker"],
        "company_name": company["company_name"],
        "short_name": company["short_name"],
        "exchange": company["exchange"],
        "quote_type": company["quote_type"],
        "query": final_query,
        "start_date": start_date,
        "end_date": end_date,
        "window_days": window_days,
        "number_of_windows": len(windows),
        "article_count_after_deduplication": len(articles_df),
        "request_log": pd.DataFrame(request_log)
    }

    return articles_df, metadata

In [ ]:
# ============================================================
# CELL 7 — User inputs
# ============================================================

# Change these values.
#TICKER = "AAPL"
#START_DATE = "2026-01-01"
#END_DATE = "2026-01-21"

TICKER = input("Enter the Yahoo Finance ticker, for example AAPL: ").strip().upper()
START_DATE = input("Enter start date in YYYY-MM-DD format: ").strip()
END_DATE = input("Enter end date in YYYY-MM-DD format: ").strip()

# Example:
# ticker = "AAPL"
# start_date = "2020-07-01"
# end_date = "2026-07-16"

# Add brand names or common company-name variations.
COMPANY_ALIASES = [
    "Apple"
]

# Leave empty to collect general company coverage.
# Add terms to focus on reputation-related events.
ADDITIONAL_TERMS = [
    # "lawsuit",
    # "cybersecurity",
    # "data breach",
    # "recall",
    # "investigation",
    # "controversy"
]

articles_df, search_metadata = get_gdelt_company_articles(
    ticker=TICKER,
    start_date=START_DATE,
    end_date=END_DATE,
    aliases=COMPANY_ALIASES,
    additional_terms=ADDITIONAL_TERMS,

    # Optional filters:
    source_country=None,       # Example: "US"
    source_language=None,      # Example: "English"

    # Usually leave False because ticker symbols can be ambiguous.
    include_ticker_in_query=False,

    # Reduce to 1–3 days for companies receiving heavy media coverage.
    window_days=7,

    max_records_per_window=250,
    request_delay=1.0
)

print("Resolved company:", search_metadata["company_name"])
print("GDELT query:", search_metadata["query"])
print("Articles retrieved:", len(articles_df))

display(articles_df.head(10))

Enter the Yahoo Finance ticker, for example AAPL: AAPL
Enter start date in YYYY-MM-DD format: 2026-07-16
Enter end date in YYYY-MM-DD format: 2026-08-11


Retrieving GDELT articles for AAPL:   0%|          | 0/4 [00:00<?, ?it/s]

Resolved company: Apple Inc.
GDELT query: ("Apple Inc." OR "Apple")
Articles retrieved: 750


,ticker,company_name,seendate,title,url,domain,language,sourcecountry,socialimage,search_query,query_window_start,query_window_end,url_mobile
0,AAPL,Apple Inc.,2026-07-22 22:30:00+00:00,Gov . Greg Abbott signs executive order target...,https://www.sacurrent.com/news/gov-greg-abbott...,sacurrent.com,English,United States,https://www.sacurrent.com/wp-content/uploads/2...,"(""Apple Inc."" OR ""Apple"")",2026-07-16,2026-07-22 23:59:59,
1,AAPL,Apple Inc.,2026-07-22 22:30:00+00:00,Constitution Enforcement : 6 Ignored Steps Tha...,https://blog.tenthamendmentcenter.com/2026/07/...,blog.tenthamendmentcenter.com,English,United States,https://blog.tenthamendmentcenter.com/files/20...,"(""Apple Inc."" OR ""Apple"")",2026-07-16,2026-07-22 23:59:59,
2,AAPL,Apple Inc.,2026-07-22 22:30:00+00:00,Into the fold : tech rivals to bet big on flex...,https://www.maitlandmercury.com.au/story/93159...,maitlandmercury.com.au,English,Australia,https://www.maitlandmercury.com.au/images/tran...,"(""Apple Inc."" OR ""Apple"")",2026-07-16,2026-07-22 23:59:59,
3,AAPL,Apple Inc.,2026-07-22 22:30:00+00:00,Comic - Con 2026 : Marvel returns to Hall H an...,https://lethbridgeherald.com/entertainment/ent...,lethbridgeherald.com,English,United States,,"(""Apple Inc."" OR ""Apple"")",2026-07-16,2026-07-22 23:59:59,
4,AAPL,Apple Inc.,2026-07-22 22:30:00+00:00,Lotmentcapitalって使いやすい ？ 実際に使ったからこそ言える本音 - メンタル整理,https://blog.udn.com:443/80ac8e58/191535278,blog.udn.com,Japanese,Taiwan,https://g.udn.com.tw/upfiles/B_C3/c303e0eb/PSN...,"(""Apple Inc."" OR ""Apple"")",2026-07-16,2026-07-22 23:59:59,
5,AAPL,Apple Inc.,2026-07-22 22:30:00+00:00,Mamdani Netanyahu backtrack shows NYC mayor di...,https://nypost.com/2026/07/22/us-news/mamdanis...,nypost.com,English,United States,https://nypost.com/wp-content/uploads/sites/2/...,"(""Apple Inc."" OR ""Apple"")",2026-07-16,2026-07-22 23:59:59,
6,AAPL,Apple Inc.,2026-07-22 22:30:00+00:00,Kimi K3让美国羡慕又不甘 : 投入8倍 为啥仍是中国先做出来 ?,https://www.vava8.com/index.php?app=index&act=...,vava8.com,Chinese,,https://img.beimg.com/data/2026/07/22/pic_4170...,"(""Apple Inc."" OR ""Apple"")",2026-07-16,2026-07-22 23:59:59,
7,AAPL,Apple Inc.,2026-07-22 22:30:00+00:00,Apple Rumored Upgrades To Its Mac Lineup Go Be...,https://www.engadget.com/2221027/apples-rumore...,engadget.com,English,Japan,https://www.engadget.com/img/gallery/apples-ru...,"(""Apple Inc."" OR ""Apple"")",2026-07-16,2026-07-22 23:59:59,
8,AAPL,Apple Inc.,2026-07-22 22:30:00+00:00,Would a MacBook Neo with more RAM just defy th...,https://www.stuff.tv/news/would-a-macbook-neo-...,stuff.tv,English,United Kingdom,https://www.stuff.tv/wp-content/uploads/sites/...,"(""Apple Inc."" OR ""Apple"")",2026-07-16,2026-07-22 23:59:59,
9,AAPL,Apple Inc.,2026-07-22 22:30:00+00:00,The latest thing in cocktails is spicy and sav...,https://more961.com/news/030030-the-latest-thi...,more961.com,English,United States,https://news.sagacom.com/wp-content/blogs.dir/...,"(""Apple Inc."" OR ""Apple"")",2026-07-16,2026-07-22 23:59:59,


In [ ]:
# ============================================================
# CELL 8 — Review request coverage
# ============================================================

request_log_df = search_metadata["request_log"]

display(request_log_df)

# Windows returning exactly 250 records may have reached the API cap.
potentially_truncated = request_log_df[
    request_log_df["records_returned"] >= 250
]

if not potentially_truncated.empty:
    print(
        "Warning: Some windows returned the maximum number of records. "
        "Run the extraction again with a smaller window_days value, "
        "such as 1 or 2 days."
    )

    display(potentially_truncated)

,window_start,window_end,records_returned,status,error
0,2026-07-16,2026-07-22 23:59:59,250,success,NaN
1,2026-07-23,2026-07-29 23:59:59,250,success,NaN
2,2026-07-30,2026-08-05 23:59:59,0,failed,GDELT request failed.
3,2026-08-06,2026-08-11 23:59:59,250,success,NaN


,window_start,window_end,records_returned,status,error
0,2026-07-16,2026-07-22 23:59:59,250,success,NaN
1,2026-07-23,2026-07-29 23:59:59,250,success,NaN
3,2026-08-06,2026-08-11 23:59:59,250,success,NaN


In [ ]:
# ============================================================
# CELL 9 — Save the dataset
# ============================================================

output_directory = Path("/content/drive/MyDrive/ColabNotebooks/DataForReputation")
output_directory.mkdir(parents=True, exist_ok=True)

safe_company_name = re.sub(
    r"[^A-Za-z0-9_-]+",
    "_",
    search_metadata["company_name"]
).strip("_")

date_label = f"{START_DATE}_to_{END_DATE}"

csv_path = (
    output_directory
    / f"{TICKER}_{safe_company_name}_{date_label}_gdelt_articles.csv"
)

parquet_path = (
    output_directory
    / f"{TICKER}_{safe_company_name}_{date_label}_gdelt_articles.parquet"
)

request_log_path = (
    output_directory
    / f"{TICKER}_{date_label}_gdelt_request_log.csv"
)

articles_df.to_csv(
    csv_path,
    index=False,
    encoding="utf-8-sig"
)

articles_df.to_parquet(
    parquet_path,
    index=False
)

request_log_df.to_csv(
    request_log_path,
    index=False
)

print(f"CSV saved to: {csv_path}")
print(f"Parquet saved to: {parquet_path}")
print(f"Request log saved to: {request_log_path}")

CSV saved to: /content/drive/My Drive/ColabNotebooks/DataForReputation/AAPL_Apple_Inc_2026-07-16_to_2026-08-11_gdelt_articles.csv
Parquet saved to: /content/drive/My Drive/ColabNotebooks/DataForReputation/AAPL_Apple_Inc_2026-07-16_to_2026-08-11_gdelt_articles.parquet
Request log saved to: /content/drive/My Drive/ColabNotebooks/DataForReputation/AAPL_2026-07-16_to_2026-08-11_gdelt_request_log.csv
